# Basic run for geodesic distance calculation on brain 

Use the scikit-surgery and VTK to visualize the mesh and create a GUI. To calculate the geodesic distance, I will use `pygeodesic`. `pygeodesic` calculates the path and distance between two points. The Python library is also available on github: https://github.com/mhogg/pygeodesic

In [1]:
import os
import platform

import pygeodesic
import pygeodesic.geodesic as geodesic

# sksurgery imports
from sksurgeryimage.utilities.utilities import are_similar
import sksurgeryvtk.models.vtk_surface_model as sm
import sksurgeryvtk.models.vtk_point_model as pm  
import sksurgeryvtk.models.vtk_sphere_model as sphm  
import numpy as np
import vtk

# application creation
from PySide6.QtWidgets import QApplication
from sksurgeryvtk.widgets.vtk_overlay_window import VTKOverlayWindow

### Create a path to redirect the errors while running the script

In [2]:
def create_vtk_error_redirect(output_path="tests/output/vtk.err.txt"):
    """Redirect VTK errors to a file (like your fixture)."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    err_out = vtk.vtkFileOutputWindow()
    err_out.SetFileName(output_path)
    vtk_std_err = vtk.vtkOutputWindow()
    vtk_std_err.SetInstance(err_out)
    return vtk_std_err

### Create a VTK overlay window, what does it mean?

In [3]:
def create_overlay_window():
    """Create VTKOverlayWindow with same init_widget behavior as fixture."""
    if platform.system() == "Linux":
        init_widget_flag = False
    else:
        init_widget_flag = True
    vtk_overlay = VTKOverlayWindow(offscreen=False, init_widget=init_widget_flag)
    return vtk_overlay


In [26]:
def smooth_model(model, iterations = 100, pass_band = 0.001):
    model_volume = model.get_volume()
    original_data = model.get_vtk_source_data()
    # Apply smoothing
    smoother = vtk.vtkWindowedSincPolyDataFilter()
    smoother.SetInputData(original_data)
    smoother.SetNumberOfIterations(iterations)
    smoother.SetPassBand(pass_band)
    smoother.NormalizeCoordinatesOn()
    smoother.Update()
    smoothed_data = smoother.GetOutput()

    smoothed_model = sm.VTKSurfaceModel(None,  (1.0, 1.0, 1.0))  # create a new VTKSurfaceModel with the smoothed data
    smoothed_model.set_source(smoothed_data)

    # Compute volume ratio and scale factor
    smoothed_volume = smoothed_model.get_volume()
    scale_factor = 1 * ((model_volume / smoothed_volume) ** (1/3))  # cube root because 3D ## model can be divided into multiple parts to check the scaling needs for each of them.
    print(f"Original volume: {model_volume:.2f}")
    print(f"Smoothed volume: {smoothed_volume:.2f}")
    print(f"Scale factor:    {scale_factor:.4f}")

    # Rescale the smoothed mesh
    transform = vtk.vtkTransform()
    transform.Scale(scale_factor, scale_factor, scale_factor)

    transformer = vtk.vtkTransformPolyDataFilter()
    transformer.SetInputData(smoothed_data)
    transformer.SetTransform(transform)
    transformer.Update()

    rescaled_data = transformer.GetOutput()

    expanded_model = sm.VTKSurfaceModel(None,  (1.0, 1.0, 1.0))  # create a new VTKSurfaceModel with the smoothed data
    expanded_model.set_source(rescaled_data)
    print(f"Rescaled volume: {expanded_model.get_volume():.2f}")

    return expanded_model


In [31]:
def run_overlay_liver_points(overlay, app, sourceIdx=5000, targetIdx=2000):
    output_name = "tests/output/"
    os.makedirs(output_name, exist_ok=True)

    ## Loads a simple STL model of brain and creates an object of tye VTKSurfaceModel. 
    ## Filename and colour are required arguments. 
    ## Get the VTKpolydata by using model.get_vtk_source_data()
    model = sm.VTKSurfaceModel("brain_models/brain_simple.stl", (1.0, 1.0, 1.0))
    print(type(model.get_vtk_source_data()))

    # image = cv2.imread(input_image_file)

    width = 640
    height = 480


    screen = app.primaryScreen()
    while width >= screen.geometry().width() or height >= screen.geometry().height():
        width //= 2
        height //= 2

    print(f"Width should be {width}, height should be {height}.")


    ### Apply smoothing to the model. This is just to make the geodesic path look better.
    # Apply smoothing, windowed sinc filter
    smoothed_model = smooth_model(model, iterations=400, pass_band=0.001)

    ### Get points and triangles from the smoothed model to calculate geodesic path. With smoothing, the vertex gets its position updated by averaging it with the neighbors.
    smoothed_points, smoothed_triangles = smoothed_model.getPointsAndCellsFromPolydata()
    print(f"Number of points in smoothed model: {smoothed_points.shape[0]}")
    print(f"Number of triangles in smoothed model: {smoothed_triangles.shape[0]}")
    ## Calculation of geodesic path on the brain surface.
    points, triangles = model.getPointsAndCellsFromPolydata()
    print(f"Number of points in original model: {points.shape[0]}")
    print(f"Number of triangles in original model: {triangles.shape[0]}")

    # geobrain = geodesic.PyGeodesicAlgorithmExact(points, triangles)
    geobrain  = geodesic.PyGeodesicAlgorithmExact(smoothed_points, smoothed_triangles)

    distance, path = geobrain.geodesicDistance(sourceIdx, targetIdx)
    
    # Create actors
    # polydata_actor = createPolyDataActor(polydataFromPointsAndCells(points, faces))
    vtkpoints = pm.VTKPointModel(path,  np.full(path.shape, 0, dtype=np.uint8))
    vtkpoints.set_point_size(10)
    
    vtkspheres = sphm.VTKSphereModel(smoothed_points[(sourceIdx,targetIdx),:], radius=3, colour= (1.0, 0.0, 0.0))
    overlay.add_vtk_models([model, vtkpoints, vtkspheres])

    # widget_vtk_overlay.add_vtk_models([model])
    # widget_vtk_overlay.set_video_image(image)
    # widget_vtk_overlay.set_camera_pose(np.linalg.inv(model_to_camera))
    overlay.show()
    overlay.resize(width, height)
    overlay.AddObserver("ExitEvent", lambda o, e, a=app: a.quit())

    print(f"Width is {overlay.width()}, height is {overlay.height()}.")

    # Run event loop so you can see the window
    app.exec()

    overlay.close()

In [33]:
# Create the Qapplication
app = QApplication.instance()
if app is None:
    app = QApplication([])

# Redirect VTK errors like the fixture did
create_vtk_error_redirect("tests/output/vtk.err.txt")

# Create overlay window
overlay = create_overlay_window()
model  = run_overlay_liver_points(overlay, app)

<class 'vtkmodules.util.data_model.PolyData'>
Width should be 640, height should be 480.
Original volume: 379842.05
Smoothed volume: 323475.07
Scale factor:    1.0550
Rescaled volume: 379842.05
Number of points in smoothed model: 33248
Number of triangles in smoothed model: 66492
Number of points in original model: 33248
Number of triangles in original model: 66492
Width is 640, height is 480.


#### Get points and triangles for the mesh data from brain. 

### Initialize the Geodesic algorithm class and get the distance between source and target points

In [7]:
# Create actors
# polydata_actor = createPolyDataActor(polydataFromPointsAndCells(points, faces))
vtkpoints = pm.VTKPointModel(path,  np.full(path.shape, 0, dtype=np.uint8))
vtkpoints.set_point_size(10)
overlay.remove_all_models()

# path_actor = createPolyLineActor(path, color=(1,1,1))
# point_actors = [createSphereActor(points[indx], radius=0.001) for indx in [sourceIndex, targetIndex]]

NameError: name 'path' is not defined

In [ ]:
overlay.add_vtk_models([model, vtkpoints])
overlay.show()
overlay.resize(640,480)
overlay.AddObserver("ExitEvent", lambda o, e, a=app: a.quit())

print(f"Width is {overlay.width()}, height is {overlay.height()}.")

# Run event loop so you can see the window
app.exec()

overlay.close()

Width is 640, height is 480.


True